# G.A.S.P. onboarding: EIA weekly storage

You'll pull weekly US natural gas storage data from the EIA API, clean it in pandas, and plot this year against the last 5 years.

You write most of the code. Every step tells you what to do and exactly where in the docs to look. Checkpoints tell you if you got it right.

**Don't edit this file.** Make your own copy in Step 0.

## Step 0: setup

Open a terminal in the repo folder (VS Code: Terminal > New Terminal).

**1. Make your own branch** so you never work on main:
```bash
git switch main                          # go to main
git pull                                 # get the latest version of it
git switch -c onboarding/yourname        # make your branch and switch to it
```

**2. Make a virtual environment.** It's a Python install just for this project, so everyone has the same package versions:
```bash
python3 -m venv .venv                    # creates the .venv folder (windows: python -m venv .venv)
source .venv/bin/activate                # turns it on (windows: .venv\Scripts\activate)
```
You should now see `(.venv)` at the start of your terminal line.

**3. Install the project's packages** (pandas, requests, matplotlib, etc.):
```bash
pip install -r requirements.txt          # installs every package listed in requirements.txt
```

**4. Open the notebook**
- in VS Code, install the **Python** and **Jupyter** extensions
- copy this file to `notebooks/onboarding/submissions/yourname.ipynb` and open your copy
- click **Select Kernel** (top right) > **Python Environments** > pick `.venv`

## Step 1: get an EIA API key

1. Register at https://www.eia.gov/opendata/register.php. It's free and the key gets emailed to you right away.
2. In the repo root, copy `.env.example` and name the copy `.env`
3. Open `.env` and paste your key after the `=` with no quotes or spaces: `EIA_API_KEY=yourkeyhere`
4. Check the Source Control panel in VS Code. `.env` should **not** be listed. If it is, stop and tell a PM.

`.env` never gets committed. Your code reads the key from that file so it's never typed into the notebook. Everyone uses their own key.

## Step 2: load your key

Run this cell. It should print the first 4 characters of your key. If you get an error, either `.env` isn't in the repo root or the name in it isn't exactly `EIA_API_KEY`.

In [ ]:
import os
import requests
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv()                           # parses your .env file and loads the variables found within
API_KEY = os.getenv("EIA_API_KEY")      # gets your api key from your .env
print(API_KEY[:4])                      # prints first four letters of your API key if it found it

## Docs you'll use

- **EIA API docs:** https://www.eia.gov/opendata/documentation.php
- **requests quickstart:** https://docs.python-requests.org/en/latest/user/quickstart/
- **pandas:** linked in each step

Some section titles on the EIA docs page don't line up with the text under them. So each step gives you a phrase to search for instead. Use **Cmd+F** (mac) or **Ctrl+F** (windows) on the page.

## Step 3: build your query in the API browser

Go to https://www.eia.gov/opendata/browser/ and set:

- **Route:** Natural Gas > Storage > the weekly working gas route
- **Frequency:** weekly
- **Data type:** value (already checked)
- **Filter by facet > Series:** check only **Weekly Lower 48 States Natural Gas Working Underground Storage** (`NW2_EPG0_SWO_R48_BCF`)
- **Sort / order:** time period, ascending

Click **Submit**, copy the **API URL**, and paste it in the cell below.

Checkpoint: your URL contains `facets[series][]=NW2_EPG0_SWO_R48_BCF` and `sort[0][direction]=asc`. If it says `desc`, change it to `asc` by hand.

In [ ]:
# paste your API URL here as a comment, you'll use it in Steps 4 and 5
#

## Step 4: understand every piece of your URL

Before you write any code, look up what each part of your URL does. Search the [EIA docs](https://www.eia.gov/opendata/documentation.php) for:

| part of your URL | search the EIA docs for |
|---|---|
| `/v2/natural-gas/stor/wkly` | `tree-like hierarchy` |
| `/data/` at the end | `final node` |
| `frequency=weekly` | `frequency query parameter` |
| `data[0]=value` | `data[0]=price` |
| `facets[series][]=...` | `Facets enable us to filter` |
| `sort[0][column]=period` and `sort[0][direction]=asc` | `SORT_PRECEDENCE` |
| `offset=0` and `length=5000` | `page through` |

Notice the browser URL has **no API key**. Search the docs for `Using an API key` to see how it goes in.

## Step 5: send the request

In Python you don't type out the whole URL. You split it:

- everything **before** the `?` is your `url`
- every `name=value` **after** the `?` (they're separated by `&`) becomes `"name": "value"` in a `params` dict
- your API key goes in `params` too

`requests` puts them back together into the same URL for you. Read **Passing Parameters In URLs** in the [requests quickstart](https://docs.python-requests.org/en/latest/user/quickstart/).

Then:
- if the request failed (bad key, typo in the url), make it raise an error. requests quickstart: **Response Status Codes**
- turn the response into a Python dict. requests quickstart: **JSON Response Content**
- pull out the list of rows. To see where the rows live in a response, search the EIA docs for `7440` and look at the example response there

Checkpoints:
- `len(rows)` equals `int(response["response"]["total"])`. `total` comes back as text, like every value (EIA docs: search `as strings`)

In [ ]:
url = ""                                 # TODO: everything before the ? in your browser url

params = {
    # TODO: your api key
    # TODO: every name=value after the ? goes here as "name": "value"
}

r = ...                                  # TODO: send the GET request with url and params
# TODO: raise an error if the request failed

response = ...                           # TODO: turn the response into a python dict
rows = ...                               # TODO: the list of rows inside response

len(rows)

## Step 6: get it into pandas

1. Make a DataFrame from `rows`. `rows` is a list of dicts, see **From a list of dicts** in [Intro to data structures](https://pandas.pydata.org/docs/user_guide/dsintro.html)
2. Keep only the `period` and `value` columns, see **How do I select specific columns** in [this tutorial](https://pandas.pydata.org/docs/getting_started/intro_tutorials/03_subset_data.html)
3. `period` is text. Convert it with [`pd.to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
4. `value` is **also text** (EIA docs: search `as strings`). Convert it with [`pd.to_numeric`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html)

Checkpoints:
- `df.dtypes` shows `datetime64` for period and a number type for value
- `df["period"].dt.day_name().value_counts()` shows only Friday. Each date is the last day of an EIA reporting week

In [ ]:
df = ...                                 # TODO: dataframe from rows, keep only period and value

df["period"] = ...                       # TODO: turn "period" column which is in text to datetime
df["value"] = ...                        # TODO: turn "value" column which is in text to numbers

df.dtypes

## Step 7: weekly change

`value` is the total gas in storage, in Bcf (billion cubic feet). EIA's weekly report is about the **change** from the week before, and that change is what this project forecasts.

Add a `change` column: this week's value minus last week's. Use [`Series.diff`](https://pandas.pydata.org/docs/reference/api/pandas.Series.diff.html).

Checkpoints:
- positive in summer (gas going into storage, called **injections**)
- negative in winter (gas coming out, called **withdrawals**)
- the first row is `NaN` since there's no week before it
- if the signs look flipped, your data is sorted newest first. Go back to Step 3

In [ ]:
df["change"] = ...                       # TODO: this week's value minus last week's

df.tail(10)

## Step 8: the 5-year range

For each week of the year, find the min, max, and average storage level across the 5 full years before this one.

1. Get ISO `year` and `week` columns from `period` with [`.dt.isocalendar()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.isocalendar.html). It returns a DataFrame with `year`, `week`, and `day` columns. Use its `year`, not `.dt.year`: weeks near New Year can belong to the year before (the example on that page shows 2010-01-01 as week 53 of 2009)
2. `current_year` is the latest year in `df`
3. `this_year` is the rows where year is `current_year`. See **How do I filter specific rows** in [the same tutorial](https://pandas.pydata.org/docs/getting_started/intro_tutorials/03_subset_data.html)
4. `last5` is the rows from the 5 years before `current_year`. Use `.isin()` from that same tutorial section with `range(current_year - 5, current_year)` (plain Python: `range(a, b)` gives a up to b - 1)
5. `band` is `last5` grouped by `week`, with the `"min"`, `"max"`, and `"mean"` of `value`. See the **Multiple aggregations** example on the [`DataFrameGroupBy.agg`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.agg.html) page

Checkpoint: `band` has 52 rows and columns named `min`, `max`, `mean`. The plot in Step 9 uses exactly these names, and `this_year` needs `week` and `value` columns.

In [ ]:
iso = ...                                # TODO: year, week, day from period
df["year"] = ...                         # TODO: iso year
df["week"] = ...                         # TODO: iso week

current_year = ...                       # TODO: latest year in df

this_year = ...                          # TODO: rows where year is current_year
last5 = ...                              # TODO: rows from the 5 years before current_year

band = ...                               # TODO: group last5 by week, min/max/mean of value

band

## Step 9: plot

Done for you. If Step 8 is right, this runs as is.

Once it runs, look at the chart. Where does this year's line slope down? Those are the weeks where the weekly change is negative (withdrawals). Find the week it bottoms out and the week it starts climbing again, then check those same weeks in your `change` column from Step 7.

In [ ]:
plt.figure(figsize=(11, 5))                                          # sets the size of the chart

plt.fill_between(band.index.astype(int), band["min"], band["max"],   # shades between the 5 year min and max
                 alpha=0.3, label="5-year range")
plt.plot(band.index.astype(int), band["mean"],                       # dashed line for the 5 year average
         linestyle="--", label="5-year average")
plt.plot(this_year["week"].astype(int), this_year["value"],          # solid line for this year
         linewidth=2, label=str(current_year))

plt.xlabel("Week of year")
plt.ylabel("Working gas in storage (Bcf)")
plt.title("Lower 48 working gas in storage vs. 5-year range")
plt.legend()                                                         # shows the labels
plt.show()

## Step 10: submit

1. Restart the kernel and click **Run All**. It should run top to bottom with no errors.
2. Commit **only your notebook**:
```bash
git add notebooks/onboarding/submissions/yourname.ipynb   # stage only your notebook
git commit -m "chore: onboarding notebook (yourname)"
git push -u origin onboarding/yourname                    # pushes your branch to GitHub
```
3. Open the link git prints, create a pull request into `main`, and put `Closes #<issue number>` in the description.

Never commit `.env`. Step 2 prints 4 characters of your key, which is fine. Never print the whole thing.